In [ ]:
print("Prepering dependecies.")
!pip -q install --upgrade unsloth trl peft accelerate bitsandbytes
print("Dependencies installed.")

In [ ]:
import torch

print("=== Runtime info ===")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"GPU 0: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"GPU 0 memory (GB): {props.total_memory / 1e9:.2f}")
else:
    print("GPU: None")

In [ ]:
from pathlib import Path
import os

# Needed for continual pretraining (disables CCE which is unsupported for CPT).
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["TORCHINDUCTOR_DISABLE"] = "1"

PRE_TRAIN_DIR = Path("/kaggle/input/datasets/victorgrigoras/llm-fine-tunning/pre-train.json")
FINE_TUNE_DIR = Path("/kaggle/input/datasets/victorgrigoras/llm-fine-tunning/train.json")

print(f"Pre-train path: {PRE_TRAIN_DIR} (exists={PRE_TRAIN_DIR.exists()})")
print(f"Fine-tune path: {FINE_TUNE_DIR} (exists={FINE_TUNE_DIR.exists()})")

In [ ]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit"
max_seq_length = 1024
dtype = None

print("Loading model...")
print(f"Model: {model_name}")
print(f"Max seq length: {max_seq_length}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True,
 )
print("Model loaded.")

In [ ]:
from datasets import load_dataset

print("Loading datasets...")
pre_train_dataset = load_dataset("json", data_files=str(PRE_TRAIN_DIR), split="train")
fine_tune_dataset = load_dataset("json", data_files=str(FINE_TUNE_DIR), split="train")

print(f"Pre-training dataset size: {len(pre_train_dataset)}")
print(f"Pre-training columns: {pre_train_dataset.column_names}")
print(f"Fine-tuning dataset size: {len(fine_tune_dataset)}")
print(f"Fine-tuning columns: {fine_tune_dataset.column_names}")

In [ ]:
print("Configuring LoRA...")
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=True,
)
print("LoRA configured.")

In [ ]:
MAX_STEPS=3500
LR=1.2e-4
WARMUP_STP=50
MAX_NORM=0.68

In [ ]:
from unsloth import UnslothTrainer, UnslothTrainingArguments

EOS_TOKEN = tokenizer.eos_token

def formatting_pretrain_func(examples):
    sentences = examples["sentence"]
    if isinstance(sentences, list):
        return {"text": [f"{s}{EOS_TOKEN}" for s in sentences]}
    return {"text": [f"{sentences}{EOS_TOKEN}"]}

print("Formatting pre-training dataset...")
pre_train_dataset = pre_train_dataset.map(formatting_pretrain_func, batched=True)
print("Pre-training dataset formatted.")

print("Setting up pre-training trainer...")
trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=pre_train_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=UnslothTrainingArguments(
        max_steps=MAX_STEPS,
        max_grad_norm=MAX_NORM,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=WARMUP_STP,
        num_train_epochs=1,
        learning_rate=LR,
        embedding_learning_rate=1e-5,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs_pretrain",
        report_to="none",
        save_steps=200,
    ),
)
print("Pre-training trainer ready.")

In [ ]:
print("Starting pre-training...")
trainer_stats = trainer.train()
print("Pre-training complete.")

In [ ]:
# Fine-tuning phase (instruction tuning)

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = (
            "Below is an instruction that describes a task, paired with an input that provides "
            "further context. Write a response that appropriately completes the request.\n\n"
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input}\n\n"
            f"### Response:\n{output}{tokenizer.eos_token}"
        )
        texts.append(text)
    return {"text": texts}

print("Formatting fine-tune dataset...")
fine_tune_dataset = fine_tune_dataset.map(formatting_prompts_func, batched=True)
print("Fine-tune dataset formatted.")

print("Setting up fine-tuning trainer...")
trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=fine_tune_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=UnslothTrainingArguments(
        max_steps=MAX_STEPS,
        max_grad_norm=MAX_NORM,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=WARMUP_STP,
        num_train_epochs=1,
        learning_rate=LR,
        embedding_learning_rate=1e-5,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs_finetune",
        report_to="none",
        save_steps=200,
    ),
)

In [ ]:
print("Starting fine-tuning...")
trainer.train()
print("Fine-tuning complete.")

In [ ]:
import torch
FastLanguageModel.for_inference(model)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device for inference: {device}")

def generate_response(prompt_text):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": "You are a helpful assistant."}]},
        {"role": "user", "content": [{"type": "text", "text": prompt_text}]},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        use_cache=True,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

    input_length = inputs.input_ids.shape[1]
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    return response

print("Test 1 (Capital):")
print(generate_response("Care-i capitala Franței?"))
print("-" * 30)

print("Test 2 (Recipe):")
print(generate_response("Dă-mi o rețetă de gogoși cu prune."))

In [ ]:
import os
import shutil

targets = ["outputs_finetune", "outputs_pretrain", "unsloth_compiled_cache"]

for name in targets:
    path = os.path.join(os.getcwd(), name)  # adjust base path if needed
    if os.path.isdir(path):
        shutil.rmtree(path)
        print(f"Deleted: {path}")
    else:
        print(f"Not found: {path}")

In [ ]:
from pathlib import Path
from unsloth import FastLanguageModel

print("Saving LoRA and GGUF outputs...")
FastLanguageModel.for_training(model)

output_root = Path("outputs")
output_root.mkdir(exist_ok=True)

lora_dir = output_root / "lora_model"
gguf_dir = output_root / "gguf_model"

model.save_pretrained(str(lora_dir))
tokenizer.save_pretrained(str(lora_dir))

try:
    gguf_model, gguf_tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(lora_dir),
        max_seq_length=max_seq_length,
        dtype=dtype,
        load_in_4bit=True,
    )
    gguf_model.save_pretrained_gguf(
        str(gguf_dir),
        gguf_tokenizer,
        quantization_method="q4_k_m",
    )
    print(f"GGUF saved: {gguf_dir}")
except Exception as exc:
    print(f"GGUF export failed: {exc}")

print(f"LoRA saved: {lora_dir}")